# Darcy-CNN Profiling

In [1]:
# cd("/content")
# isdir("/content/darcy-cnn")
# rm("/content/darcy-cnn", recursive=true, force=true)
# run(`git clone https://github.com/thezettascale/darcy-cnn.git /content/darcy-cnn`)
# cd("/content/darcy-cnn")
# @assert isfile("Project.toml")
# println("Fresh clone ready at ", pwd())

# # Cache packages to Google Drive if mounted
# drive_depot = joinpath(ENV["HOME"], "drive", "MyDrive", "julia_depot")
# if isdir(dirname(drive_depot))
#     mkpath(drive_depot)
#     pushfirst!(DEPOT_PATH, drive_depot)
#     @info "Package cache: $drive_depot"
# end

# using Pkg; Pkg.activate(pwd()); Pkg.resolve(); Pkg.instantiate()

In [2]:
ENV["DEVICE"] = "gpu"

using Random, Lux, Lux.Training, Reactant, Optimisers, Statistics
using Printf, JSON
using MLDataDevices: reactant_device, cpu_device

cd("..")
include("src/WavKANConv.jl"); using .WavKANConv

In [3]:
MODEL_NAMES = ["CNN", "FNO", "KAN_CNN"]
PROFILE_BS = 2

function setup(name::String; dev=reactant_device())
    rng = Random.default_rng()
    Random.seed!(rng, 42)

    cfg = load_config(name)
    train_loader, test_loader = get_darcy_loader(PROFILE_BS; dev=dev)
    x_sample, y_sample = first(train_loader)

    model = create_model(cfg)
    ps, st = Lux.setup(rng, model)
    ps, st = ps |> dev, st |> dev
    st_test = Lux.testmode(st)
    n_params = Lux.parameterlength(model)

    loss_fn(y_pred, y) = loss_fcn(y_pred, y; p=cfg.p)
    objective(model, ps, st, (x, y)) = (loss_fn(first(model(x, ps, st)), y), last(model(x, ps, st)), (;))
    ts = Training.TrainState(model, ps, st, Optimisers.Adam(cfg.learning_rate))

    println("$name: $n_params params")
    return (; model, ps, st, st_test, cfg, n_params, ts, objective, x_sample, y_sample)
end

function hlo_op_profile(hlo_str::String)
    ops = String[]
    for line in split(hlo_str, '\n')
        m = match(r"stablehlo\.(\w+)", line)
        m !== nothing && push!(ops, m.captures[1])
    end
    total = length(ops)
    d = Dict{String,Int}()
    for x in ops; d[x] = get(d, x, 0) + 1; end
    counts = sort(collect(d); by=last, rev=true)
    return [(op, n, 100.0 * n / total) for (op, n) in counts]
end

function print_hlo_profile(name, profile)
    total = sum(x[2] for x in profile)
    @printf("\n=== %s (%d total ops) ===\n", name, total)
    @printf("%-25s %8s %8s\n", "Op", "Count", "%")
    println("-"^43)
    for (op, n, pct) in profile
        @printf("%-25s %8d %7.1f%%\n", op, n, pct)
    end
end

function profile_and_bench(name, f, args...; warmup=3, trials=10)
    hlo = string(Reactant.@code_hlo f(args...))
    profile = hlo_op_profile(hlo)
    print_hlo_profile(name, profile)

    compiled = Reactant.@compile sync=true f(args...)
    for _ in 1:warmup; compiled(args...); end
    times = [(@elapsed compiled(args...)) * 1000 for _ in 1:trials]
    @printf("%-20s  med=%.3f ms  min=%.3f ms  max=%.3f ms\n\n", name, median(times), minimum(times), maximum(times))

    return profile, times, compiled
end

profile_and_bench (generic function with 1 method)

## Profile

In [ ]:
device = ENV["DEVICE"]
all_results = Dict{String, Any}()

for name in MODEL_NAMES
    m = setup(name)

    # Profile forward pass (HLO + timing) — testmode, no BatchNorm training constraint
    fwd(x, ps, st) = m.model(x, ps, st)
    fwd_profile, fwd_times, _ = profile_and_bench("$(name)_forward", fwd, m.x_sample, m.ps, m.st_test)

    # Benchmark train step (single_train_step! compiles & caches internally, use sync=true for timing)
    Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), m.ts; sync=true)  # warmup/compile
    ts = m.ts
    ts_times = Float64[]
    for _ in 1:10
        t = @elapsed begin
            _, _, _, ts = Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), ts; sync=true)
        end
        push!(ts_times, t * 1000)
    end
    @printf("%-20s  med=%.3f ms  min=%.3f ms  max=%.3f ms\n\n",
            "$(name)_train_step", median(ts_times), minimum(ts_times), maximum(ts_times))

    all_results[name] = Dict(
        "n_params" => m.n_params,
        "forward" => Dict(
            "timing_ms" => Dict("median" => median(fwd_times), "min" => minimum(fwd_times), "max" => maximum(fwd_times)),
            "hlo_ops" => [Dict("op" => op, "count" => n, "pct" => round(pct; digits=1)) for (op, n, pct) in fwd_profile],
            "total_ops" => sum(x[2] for x in fwd_profile),
        ),
        "train_step" => Dict(
            "timing_ms" => Dict("median" => median(ts_times), "min" => minimum(ts_times), "max" => maximum(ts_times)),
        ),
    )
end

# Save
mkpath("profiles")
outfile = "profiles/profile_$(device).json"
open(outfile, "w") do io
    JSON.print(io, Dict("device" => device, "models" => all_results), 2)
end
println("Saved to $outfile")

CNN: 5982121 params


I0000 00:00:1775945216.792409  101241 service.cc:153] XLA service 0x3d54a260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775945216.792423  101241 service.cc:161]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
I0000 00:00:1775945216.792764  101241 se_gpu_pjrt_client.cc:1440] Using BFC allocator.
I0000 00:00:1775945216.792787  101241 gpu_helpers.cc:141] XLA backend allocating 2923905024 bytes on device 0 for BFCAllocator.
I0000 00:00:1775945216.792803  101241 gpu_helpers.cc:182] XLA backend will use up to 974635008 bytes on device 0 for CollectiveBFCAllocator.
I0000 00:00:1775945216.804419  101241 cuda_dnn.cc:461] Loaded cuDNN version 91400
I0000 00:00:1775945216.812944  101241 cuda_executor.cc:541] failed to allocate 2.72GiB (2923905024 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1775945216.813026  101241 cuda_executor.cc:541] failed to allocate 2.45Gi


=== CNN_forward (55 total ops) ===
Op                           Count        %
-------------------------------------------
constant                        10    18.2%
add                              7    12.7%


W0000 00:00:1775945261.036930  101241 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.06GiB (rounded to 2216385280)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1775945261.037075  101241 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1775945261.037085  101241 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 36, Chunks in use: 36. 9.0KiB allocated for chunks. 9.0KiB in use in bin. 168B client-requested in use in bin.
I0000 00:00:1775945261.037103  101241 bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 9, Chunks in use: 9. 6.0KiB allocated for chunks. 6.0KiB in use in bin. 5.3KiB client-requested in use in bin.
I0000 00:00:1775945261.037116  101241 bfc_allocator.cc:1056] Bin (1024): 	Total Chunks: 6, Chunks in use: 6. 9.0KiB allocated for chunks. 

convolution                      7    12.7%
compare                          7    12.7%
select                           7    12.7%
broadcast_in_dim                 7    12.7%
multiply                         7    12.7%
reverse                          3     5.5%
CNN_forward           med=8.788 ms  min=8.558 ms  max=8.912 ms

CNN_train_step        med=35.448 ms  min=33.442 ms  max=2961.835 ms

FNO: 4667665 params

=== FNO_forward (244 total ops) ===
Op                           Count        %
-------------------------------------------
multiply                        68    27.9%
add                             39    16.0%
broadcast_in_dim                31    12.7%
constant                        19     7.8%
dynamic_update_slice            16     6.6%
convolution                     14     5.7%
pad                             12     4.9%
tanh                            10     4.1%
slice                            8     3.3%
fft                              8     3.3%
reduce            

I0000 00:00:1775946392.847047  101241 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775946392.853545  101241 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775946392.869221  101241 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775946425.332690  101241 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775946425.339599  101241 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775946425.345710  101241 dot_merger.cc:476] Merging Dots in computation: main.36


## XLA Trace
Upload `.pb` to [Perfetto UI](https://ui.perfetto.dev) for per-op runtime breakdown.

In [ ]:
device = ENV["DEVICE"]
for name in MODEL_NAMES
    m = setup(name)

    # Warmup (compiles & caches internally)
    Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), m.ts; sync=true)

    tracefile = "profiles/$(name)_train_step_$(device)"
    Reactant.Profiler.with_profiler(tracefile) do
        ts = m.ts
        for _ in 1:20
            _, _, _, ts = Training.single_train_step!(
                AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), ts; sync=true
            )
        end
    end
    println("$name -> $tracefile/")
end